In [ ]:
# Importing the required libraries
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re
from urllib.parse import urljoin
import openpyxl

In [2]:
# URL pattern used by the website pagination
# There are 50 pages of books on the site
base_url = "https://books.toscrape.com/catalogue/page-{}.html"

# Creating an empty list to store all scraped book data
books = []

In [3]:
# Loopingthrough all pages
for page in range(1, 51):

    # Page URL
    url = base_url.format(page)

    # Sending http request to website
    response = requests.get(url)

    # Stopping execution of the page does not exist
    if response.status_code != 200:
        print(f"page {page} not found.")
        break

    # Parsing html content using BeautifulSoup
    soup = BeautifulSoup(response.text, "html.parser")

    # Finding all books containers on the page
    book_items = soup.select("article.product_pod")

    # Looping through each on the page
    for book in book_items:

        #Etracting book title
        title = book.h3.a["title"]

        #Extracting price
        price = book.select_one("p.price_color").text.strip()

        #Extracting rating
        rating = book.select_one("p.star-rating")["class"][1]

        #Extracting stock availability
        availability = (book.select_one("p.instock.availability").text.strip())

        # Constructing url for book details page
        book_url = urljoin(url, book.h3.a["href"])

        #Requesting book details
        book_response = requests.get(book_url)

        #Parsing book for details
        book_soup = BeautifulSoup(book_response.text, "html.parser")

        #Extracting Genre
        genre = book_soup.select("ul.breadcrumb li a")[2].text.strip()
        
        #Extract Stock count
        stock_text = (
            book_soup.select_one(
                "p.instock.availability"
            )
            .text
            .strip()
        )

        stock_match = re.search(
            r"\((\d+) available\)",
            stock_text
        )

        stock_count = (
            int(stock_match.group(1))
            if stock_match
            else 0
        )

        #Storee the data
        # Store data
        books.append({
            "Title": title,
            "Genre": genre,
            "Price": price,
            "Rating": rating,
            "Availability": availability,
            "Stock_Count": stock_count
        })


In [4]:
#Converting the list to df
df = pd.DataFrame(books)

In [5]:
df.head()

,Title,Genre,Price,Rating,Availability,Stock_Count
0,A Light in the Attic,Poetry,Â£51.77,Three,In stock,22
1,Tipping the Velvet,Historical Fiction,Â£53.74,One,In stock,20
2,Soumission,Fiction,Â£50.10,One,In stock,20
3,Sharp Objects,Mystery,Â£47.82,Four,In stock,20
4,Sapiens: A Brief History of Humankind,History,Â£54.23,Five,In stock,20


In [8]:
df.shape

(1000, 6)

In [10]:
df.dtypes

Title             str
Genre             str
Price             str
Rating            str
Availability      str
Stock_Count     int64
dtype: object

In [14]:
df.to_excel('book_market_data.xlsx', index=False)